# RQ3 — Responsible Use: False Alarms vs Missed Warnings
*How do false alarms and missed warnings trade off across decision thresholds, and which threshold best supports responsible early warning?*

**Outputs:** `RQ3_threshold_tradeoff.csv`, `RQ3_threshold_tradeoff.pdf`

In [7]:

# ============================================================
# Shared data loading & preprocessing (Algerian Forest Fires)
# ============================================================
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13, "axes.labelsize": 11,
    "axes.grid": True, "grid.alpha": 0.3, "figure.autolayout": True,
})

OUTDIR = "/kaggle/working"          # on Kaggle this is the output folder
os.makedirs(OUTDIR, exist_ok=True)

def find_csv():
    """Find the Algerian Forest Fires csv anywhere under /kaggle/input."""
    cands = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not cands:                   # local fallback
        cands = glob.glob("**/*.csv", recursive=True)
    # prefer a file whose name mentions 'algerian' or 'forest'
    for c in cands:
        n = os.path.basename(c).lower()
        if "algerian" in n or "forest" in n or "fire" in n:
            return c
    return cands[0]

def load_algerian():
    path = find_csv()
    print("Loading:", path)
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.read().splitlines()
    # The raw UCI file mixes a title line, two region headers and a blank row,
    # so we parse it line by line rather than with a fixed-width csv reader.
    rows = [ln.split(",") for ln in lines]
    header = None
    region = "Bejaia"           # first block in the UCI file
    region_switched = False
    records, cols = [], None
    for r in rows:
        cells = [str(x).strip() for x in r]
        joined = " ".join(cells).lower()
        if "temperature" in joined and ("rh" in joined or "ws" in joined):
            cols = [c.strip() for c in cells if c.strip() != ""]
            header = cols
            if records and not region_switched:
                region = "Sidi-Bel Abbes"   # second header => second region
                region_switched = True
            continue
        if all(c == "" for c in cells):
            continue
        if "region" in joined or "dataset" in joined:   # title / region label line
            if "sidi" in joined:
                region = "Sidi-Bel Abbes"; region_switched = True
            continue
        if header is None:
            continue
        data_cells = [c for c in cells if c != ""]
        if len(data_cells) < len(cols):
            continue
        rec = dict(zip(cols, data_cells[:len(cols)]))
        rec["region"] = region
        records.append(rec)
    df = pd.DataFrame.from_records(records)
    return df

df = load_algerian()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

# Standardise the target column name
target_col = [c for c in df.columns if "class" in c.lower()]
target_col = target_col[0] if target_col else df.columns[-2]
df = df.rename(columns={target_col: "Classes"})

# Clean target -> binary (fire = 1, not fire = 0)
df["Classes"] = (df["Classes"].astype(str).str.strip().str.lower()
                 .str.replace(r"\s+", " ", regex=True))
df = df[df["Classes"].isin(["fire", "not fire"])].copy()
df["target"] = (df["Classes"] == "fire").astype(int)

# Numeric feature columns
FEATURES = ["Temperature", "RH", "Ws", "Rain",
            "FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]
FEATURES = [f for f in FEATURES if f in df.columns]
for c in FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=FEATURES + ["target"]).reset_index(drop=True)

print("Shape:", df.shape, "| Fire:", int(df.target.sum()),
      "| Not fire:", int((1-df.target).sum()))
print("Regions:", df["region"].value_counts().to_dict())
X = df[FEATURES].copy()
y = df["target"].copy()


Loading: /kaggle/input/notebooks/sudhanshu432/eda-and-fe-algerian-forest-fires-dataset/Algerian_forest_fires_cleaned_dataset.csv
Shape: (243, 17) | Fire: 137 | Not fire: 106
Regions: {'Bejaia': 243}


In [8]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                          stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=400, random_state=42).fit(X_tr, y_tr)
prob = model.predict_proba(X_te)[:, 1]

rows = []
for t in np.round(np.arange(0.1, 0.95, 0.05), 2):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    rows.append({
        "Threshold": t,
        "False_Alarms_FP": int(fp),     # predicted fire, none occurred
        "Missed_Warnings_FN": int(fn),  # missed an actual fire (costly!)
        "True_Pos": int(tp), "True_Neg": int(tn),
        "Recall": tp / (tp + fn) if (tp + fn) else 0,
        "Precision": tp / (tp + fp) if (tp + fp) else 0,
    })
tab = pd.DataFrame(rows).round(3)
tab.to_csv(f"{OUTDIR}/RQ3_threshold_tradeoff.csv", index=False)
print(tab.to_string(index=False))


 Threshold  False_Alarms_FP  Missed_Warnings_FN  True_Pos  True_Neg  Recall  Precision
      0.10                7                   0        41        25   1.000      0.854
      0.15                6                   0        41        26   1.000      0.872
      0.20                5                   0        41        27   1.000      0.891
      0.25                5                   0        41        27   1.000      0.891
      0.30                5                   0        41        27   1.000      0.891
      0.35                4                   0        41        28   1.000      0.911
      0.40                2                   0        41        30   1.000      0.953
      0.45                1                   0        41        31   1.000      0.976
      0.50                1                   0        41        31   1.000      0.976
      0.55                1                   0        41        31   1.000      0.976
      0.60                0                

In [9]:

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(tab["Threshold"], tab["False_Alarms_FP"], "o-", color="#E69F00",
        lw=2, label="False Alarms (FP)")
ax.plot(tab["Threshold"], tab["Missed_Warnings_FN"], "s-", color="#D55E00",
        lw=2, label="Missed Warnings (FN)")
ax.set_xlabel("Decision Threshold"); ax.set_ylabel("Number of Cases")
ax.set_title("RQ3: False Alarms vs Missed Warnings across Thresholds")
ax.legend(frameon=True)
ax.annotate("Lower threshold = fewer\nmissed fires, more false alarms",
            xy=(0.15, tab["Missed_Warnings_FN"].iloc[1]),
            xytext=(0.4, tab["False_Alarms_FP"].max()*0.8),
            arrowprops=dict(arrowstyle="->", alpha=0.6), fontsize=9)
fig.savefig(f"{OUTDIR}/RQ3_threshold_tradeoff.pdf", bbox_inches="tight")
print("Saved RQ3_threshold_tradeoff.pdf")


Saved RQ3_threshold_tradeoff.pdf
